# 08 — word2vec from scratch : CBOW, skip-gram et échantillonnage négatif
**Navigation** : [<< Précédent — 07. Vecteurs de mots (cooccurrence, PPMI, SVD)](07-Vecteurs-de-Mots-Coocc-PPMI-SVD.ipynb) | [Index de la série](README.md) | **08 (ce notebook)** | [Suivant — 09. fastText (subwords) >>](09-FastText-Subword-Vectors.ipynb)

Ce carnet implémente **word2vec** — l'algorithme qui a popularisé les **vecteurs de mots denses** — **de zéro**, avec le forward, la perte et le **gradient analytique** calculés à la main en C#.

Suite directe du carnet **07** : là où celui-ci factorisait la matrice de co-occurrence (PPMI + SVD) pour obtenir des vecteurs **statistiques**, ici on **apprend** des vecteurs par **descente de gradient** sur un objectif de prédiction. Le point pédagogique central : les deux approches proposent des représentations dont les voisinages peuvent être comparés avec prudence.

**CPU-only, déterministe** (seed fixée), aucune clé API.

**Version jumelle** : [Python](08-Word2Vec-CBOW-SkipGram.ipynb).

**Objectifs** : calculer et vérifier les gradients CBOW et skip-gram, entraîner et examiner des voisinages.  
**Prérequis** : C#, algèbre linéaire et probabilités. **Durée** : 90 minutes.

Le générateur C# diffère de NumPy : les voisinages et les trajectoires numériques peuvent différer malgré la même graine conceptuelle et le même protocole.


## 1. Le corpus et le vocabulaire

Un corpus français **miniature et synthétique** (même motif que le carnet 05 : provenance explicite, pas de donnée externe). Il est construit pour que la structure sémantique soit **visible** : quelques champs lexicaux (royauté, animaux, fruits, mouvement) répétés de façon régulière, si bien que des paires comme `roi/reine` ou `pomme/poire` co-occurrent de manière parallèle.

In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Net;
using System.Globalization;
using MathNet.Numerics.Distributions;
using Microsoft.DotNet.Interactive.Formatting;
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((v,w)=>w.Write(v.Content),"text/html");
string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);
var rng=new Random(42); var normal=new Normal(0,0.1,rng);
string corpus="le roi porte la couronne sur le trone\nla reine porte la couronne sur le trone\nle roi regne dans le royaume\nla reine regne dans le royaume\nle cheval court dans le champ\nla jument court dans le champ\nle cheval danse sous un arbre\nla jument danse sous un arbre\nle chat miaule pres du chien\nle chaton miaule pres du chiot\nle chien aboie dans le jardin\nle chiot aboie dans le jardin\nla pomme est un fruit rouge\nla poire est un fruit vert\nun enfant mange la pomme\nun enfant mange la poire\n";
var tokens=corpus.ToLowerInvariant().Split((char[])null,StringSplitOptions.RemoveEmptyEntries);
var vocab=tokens.Distinct().OrderBy(x=>x,StringComparer.Ordinal).ToArray();
var word2id=vocab.Select((w,i)=>(w,i)).ToDictionary(x=>x.w,x=>x.i); int V=vocab.Length,D=16;
Console.WriteLine($".NET {Environment.Version} | MathNet {typeof(Normal).Assembly.GetName().Version}");
Console.WriteLine($"tokens : {tokens.Length} | vocabulaire : {V} | {string.Join(", ",vocab)}");

Installed Packages MathNet.Numerics, 5.0.0

.NET 10.0.14 | MathNet 5.0.0.0


tokens : 98 | vocabulaire : 36 | aboie, arbre, champ, chat, chaton, cheval, chien, chiot, couronne, court, dans, danse, du, enfant, est, fruit, jardin, jument, la, le, mange, miaule, poire, pomme, porte, pres, regne, reine, roi, rouge, royaume, sous, sur, trone, un, vert


### Lecture du résultat

98 tokens pour 36 mots de vocabulaire : chaque mot n'apparaît qu'environ 2,7 fois en moyenne. C'est le compromis assumé d'un corpus miniature — assez riche pour porter quatre champs lexicaux (royauté, équidés, petits animaux, fruits) avec une structure de substitution régulière, assez petit pour s'entraîner en secondes et s'inspecter à l'œil nu.

Deux conséquences à garder pour la suite :

- les mots-outils (`le`, `la`, `dans`, `un`) servent de colle syntaxique : ils co-occurrent avec presque tout, donc ils portent peu d'information sémantique mais beaucoup de signal de fréquence ;
- le vocabulaire est trié alphabétiquement (`sorted(set(tokens))`) : l'identifiant d'un mot est sa position dans cette liste — `roi` = 28, `la` = 18, `couronne` = 8. Ce sont ces indices qu'on lira tels quels dans les paires d'entraînement de la cellule suivante, puis dans le decode des sorties.

In [2]:
var pairs=new List<(int[] Ctx,int Tgt)>();
for(int i=0;i<tokens.Length;i++) { var ctx=Enumerable.Range(Math.Max(0,i-2),Math.Min(tokens.Length,i+2)-Math.Max(0,i-2)).Where(j=>j!=i).Select(j=>word2id[tokens[j]]).ToArray(); if(ctx.Length>0) pairs.Add((ctx,word2id[tokens[i]])); }
Console.WriteLine($"Paires : {pairs.Count} ; paire 4 : [{string.Join(", ",pairs[3].Ctx)}] → {pairs[3].Tgt}");

Paires : 98 ; paire 4 : [28, 24, 8] → 18


### Lecture du résultat

Décodons l'exemple : `([28, 24, 8], 18)` se lit « contexte [roi, porte, couronne] -> cible la » — le token `la` (indice 18) est le 4e mot du corpus (« le roi porte **la** couronne… »). La fenêtre `window=2` est **asymétrique** : elle couvre `range(i-2, i+2)` hors `i`, soit 2 mots à gauche et 1 seul à droite. D'où 3 mots de contexte pour cette position, pas 4.

Deux détails de construction à voir une fois :

- **98 paires pour 98 tokens** : chaque token engendre exactement une paire, y compris le premier (sa fenêtre tronquée à `range(0, 2)` garde le mot 1) et le dernier. Le contexte moyen fait juste sous 3 mots, la fenêtre n'étant pleine qu'à distance 2 des extrémités du corpus.
- **La tokenisation ignore les fins de phrase** : `str.split()` coupe sur tout blanc, `\n` compris — les fenêtres traversent les frontières de phrases. Le dernier contexte de « le trone » contient le `la` de la phrase suivante. Sur un vrai corpus on segmenterait par phrase ; ici la structure parallèle rend le raccord peu coûteux, mais c'est un choix à connaître.

## 2. CBOW from scratch

**CBOW** (*Continuous Bag of Words*) prédit le **mot central** depuis son **contexte**. Le modèle linéaire :

- **Entrée** : la **moyenne** des vecteurs des mots de contexte, `h = mean(W[c] for c in ctx) + b`.
- **Sortie** : un score par mot du vocabulaire, `score_j = W'_j . h + b'_j`, puis un **softmax** qui donne `P(mot | contexte)`.

La matrice d'entrée `W` (une ligne par mot) est **le vecteur de mot** qu'on cherche à apprendre. Le softmax calcule des scores sur tout le vocabulaire. L'échantillonnage négatif de la section 6 change l'objectif et ne score qu'un petit échantillon, même si notre implémentation conserve des mises à jour denses.

In [3]:
record Grad(double[,] W,double[,] Wp,double[] B,double[] Bp);
class WordModel {
    public int V,D; public double[,] W,Wp; public double[] B,Bp;
    public WordModel(int v,int d,Normal normal) { V=v;D=d; W=new double[V,D];Wp=new double[V,D];B=new double[D];Bp=new double[V]; for(int i=0;i<V;i++) for(int j=0;j<D;j++) W[i,j]=normal.Sample(); for(int i=0;i<V;i++) for(int j=0;j<D;j++) Wp[i,j]=normal.Sample(); }
    public static double Sigmoid(double x) => x>=0 ? 1/(1+Math.Exp(-x)) : Math.Exp(x)/(1+Math.Exp(x));
    public static double Softplus(double x) => Math.Max(x,0)+Math.Log(1+Math.Exp(-Math.Abs(x)));
    public static double LogSumExp(double[] x) { double m=x.Max(); return m+Math.Log(x.Sum(v=>Math.Exp(v-m))); }
    public double[] Scores(double[] h) { var z=new double[V]; for(int i=0;i<V;i++) {z[i]=Bp[i];for(int j=0;j<D;j++) z[i]+=Wp[i,j]*h[j];} return z; }
    public (double,Grad) Cbow(int[] ctx,int tgt) {
        var h=(double[])B.Clone(); foreach(int c in ctx) for(int j=0;j<D;j++) h[j]+=W[c,j]/ctx.Length;
        var z=Scores(h); double lse=LogSumExp(z),loss=lse-z[tgt]; var ds=z.Select(x=>Math.Exp(x-lse)).ToArray();ds[tgt]-=1;
        var dw=new double[V,D];var dwp=new double[V,D];var db=new double[D];var dbp=(double[])ds.Clone();
        for(int i=0;i<V;i++) for(int j=0;j<D;j++) {dwp[i,j]=ds[i]*h[j];db[j]+=Wp[i,j]*ds[i];}
        foreach(int c in ctx) for(int j=0;j<D;j++) dw[c,j]+=db[j]/ctx.Length;
        return (loss,new Grad(dw,dwp,db,dbp));
    }
    public (double,Grad) Skipgram(int tgt,int[] ctx) {
        var h=new double[D];for(int j=0;j<D;j++) h[j]=W[tgt,j]+B[j];var z=Scores(h);double lse=LogSumExp(z),loss=0;
        var ds=new double[V];foreach(int c in ctx) {loss+=lse-z[c]; for(int i=0;i<V;i++) ds[i]+=Math.Exp(z[i]-lse);ds[c]-=1;}
        var dw=new double[V,D];var dwp=new double[V,D];var db=new double[D];
        for(int i=0;i<V;i++) for(int j=0;j<D;j++) {dwp[i,j]=ds[i]*h[j];db[j]+=Wp[i,j]*ds[i];}
        for(int j=0;j<D;j++) dw[tgt,j]=db[j];return (loss,new Grad(dw,dwp,db,ds));
    }
    public (double,Grad) Sgns(int tgt,int pos,int[] negs) {
        var h=new double[D];for(int j=0;j<D;j++) h[j]=W[tgt,j]+B[j];var dw=new double[V,D];var dwp=new double[V,D];var db=new double[D];var dbp=new double[V];double loss=0;
        void Accumulate(int row,bool positive) { double x=Bp[row];for(int j=0;j<D;j++) x+=Wp[row,j]*h[j];loss+=positive?Softplus(-x):Softplus(x);double g=Sigmoid(x)-(positive?1:0); dbp[row]+=g;for(int j=0;j<D;j++){dwp[row,j]+=g*h[j];db[j]+=g*Wp[row,j];} }
        Accumulate(pos,true);foreach(int n in negs) Accumulate(n,false);for(int j=0;j<D;j++) dw[tgt,j]=db[j];return (loss,new Grad(dw,dwp,db,dbp));
    }
    public void Update(Grad g,double lr) {for(int i=0;i<V;i++){Bp[i]-=lr*g.Bp[i];for(int j=0;j<D;j++){W[i,j]-=lr*g.W[i,j];Wp[i,j]-=lr*g.Wp[i,j];}} for(int j=0;j<D;j++) B[j]-=lr*g.B[j];}
}
var model=new WordModel(V,D,normal);
Console.WriteLine($"Perte initiale CBOW : {model.Cbow(pairs[3].Ctx,pairs[3].Tgt).Item1:F6} ; ln(V)={Math.Log(V):F6}");

Perte initiale CBOW : 3.626343 ; ln(V)=3.583519


### Lecture du résultat

La perte initiale de la quatrième paire vaut **3,626**, proche de `ln(36) = 3,584` : de petits poids donnent ici une distribution proche de l'uniforme. C'est un contrôle utile pour cette initialisation, pas une garantie générale sur toute distribution initiale. Les générateurs C# et NumPy diffèrent, donc la valeur exacte peut varier.

In [4]:
double CheckGradient(Func<(double,Grad)> evaluate) {
    var analytic=evaluate().Item2; double max=0,eps=1e-5;
    void Check(ref double x,double expected) {double old=x; x=old+eps;double plus=evaluate().Item1; x=old-eps;double minus=evaluate().Item1;x=old;max=Math.Max(max,Math.Abs(expected-(plus-minus)/(2*eps)));}
    for(int i=0;i<V;i++) for(int j=0;j<D;j++){Check(ref model.W[i,j],analytic.W[i,j]);Check(ref model.Wp[i,j],analytic.Wp[i,j]);}
    for(int i=0;i<D;i++) Check(ref model.B[i],analytic.B[i]);for(int i=0;i<V;i++) Check(ref model.Bp[i],analytic.Bp[i]);return max;
}
double errInit=CheckGradient(()=>model.Cbow(pairs[3].Ctx,pairs[3].Tgt));
double errRepeat=CheckGradient(()=>model.Cbow(new[]{0,0,1},1));
if(errInit>1e-4 || errRepeat>1e-4) throw new Exception("Gradient CBOW incorrect");
Console.WriteLine($"CBOW : {2*V*D+D+V} paramètres, erreur max {errInit:E3}; contexte répété {errRepeat:E3}");

CBOW : 1204 paramètres, erreur max 4.526E-011; contexte répété 4.428E-011


### Lecture du résultat

Les différences finies contrôlent les **1 204 paramètres** (`W`, `W'`, `b`, `b'`) avant entraînement ; l'écart maximal mesuré est 4,53·10⁻¹¹. Un contexte répété est également vérifié pour contrôler l'accumulation des contributions. Cet accord porte sur ces fixtures et tolérances : il corrobore le gradient sans constituer une preuve sur toutes les entrées.

## 3. Skip-gram from scratch

Skip-gram prédit chaque mot du contexte à partir du vecteur du mot central. La perte additionne une cross-entropie par voisin. En CBOW, `W` est mise à jour sur les mots du contexte, tandis que `W'` reçoit les gradients des sorties ; en skip-gram la ligne d'entrée du centre reçoit la somme des contributions. La qualité des mots rares dépend des données et de l'entraînement, sans supériorité universelle garantie.

In [5]:
var sgLoss=model.Skipgram(pairs[3].Tgt,pairs[3].Ctx).Item1;
Console.WriteLine($"Skip-gram : perte et gradient définis ; perte de la paire 4 = {sgLoss:F6}");
// Fixture extrême, sans changer le générateur ni le modèle destiné à l'entraînement.
var fixture=new WordModel(2,1,new Normal(0,0.1,new Random(9)));fixture.W[0,0]=1;fixture.W[1,0]=0;fixture.Wp[0,0]=0;fixture.Wp[1,0]=0;fixture.B[0]=0;fixture.Bp[0]=0;fixture.Bp[1]=-30;
var (extreme,extGrad)=fixture.Cbow(new[]{0,0},1);double eps=1e-5;fixture.Bp[1]=-30+eps;double plus=fixture.Cbow(new[]{0,0},1).Item1;fixture.Bp[1]=-30-eps;double minus=fixture.Cbow(new[]{0,0},1).Item1;fixture.Bp[1]=-30;
var (skipExtreme,skipGrad)=fixture.Skipgram(0,new[]{1,1});fixture.Bp[1]=-30+eps;double sp=fixture.Skipgram(0,new[]{1,1}).Item1;fixture.Bp[1]=-30-eps;double sm=fixture.Skipgram(0,new[]{1,1}).Item1;
if(!double.IsFinite(extreme) || Math.Abs(extGrad.Bp[1]-(plus-minus)/(2*eps))>1e-7 || Math.Abs(skipGrad.Bp[1]-(sp-sm)/(2*eps))>1e-7) throw new Exception("Fixture extrême incorrecte");
Console.WriteLine($"Logits extrêmes et répétitions : CBOW gradient {extGrad.Bp[1]:F6} ; skip-gram {skipGrad.Bp[1]:F6}");

Skip-gram : perte et gradient définis ; perte de la paire 4 = 10.664728


Logits extrêmes et répétitions : CBOW gradient -1.000000 ; skip-gram -2.000000


### Lecture du résultat

Le skip-gram utilise une entrée centrale et somme les pertes des voisins, y compris leurs répétitions. Les différences finies sur la fixture extrême vérifient la cohérence de la perte et du gradient ; l'entraînement softmax de ce modèle n'est pas réalisé ici. En CBOW, ce sont les lignes d'entrée du contexte qui sont mises à jour, pas celle de la cible du seul fait qu'elle est cible.

### Exercice 1 — Vérifier le gradient skip-gram

Écrivez la vérification numérique pour `W[tgt]`, `Wp`, `b` et `bp`. Le paramètre `provider` permet de tester à la fois le gradient réel et un gradient délibérément perturbé : une réponse constante ne suffit pas.

In [6]:
// Exercice 1 : vérifier numériquement le gradient du skip-gram fourni.
bool? VerifierSkipgram(int tgt,int[] ctx,Func<int,int[],(double,Grad)> provider) {
    // TODO : différences finies pour W[tgt], Wp, B, Bp.
    return null;
}
(double,Grad) Perturbe(int tgt,int[] ctx) {var (loss,g)=model.Skipgram(tgt,ctx);g.Bp[ctx[0]]+=0.1;return (loss,g);}
var ex1=VerifierSkipgram(pairs[3].Tgt,pairs[3].Ctx,model.Skipgram);
if(ex1==null) Console.WriteLine("Exercice 1 à compléter");
else Console.WriteLine(ex1==true && VerifierSkipgram(pairs[3].Tgt,pairs[3].Ctx,Perturbe)==false ? "Fixtures gradient validées" : "Réponse incorrecte");

Exercice 1 à compléter


## 4. Entraînement CBOW + courbe de perte

La descente mélange les paires à chaque époque puis applique les gradients. Nous mesurons la perte moyenne sur 60 époques ; sa baisse n'est pas garantie pour tout pas ou tout corpus.

In [7]:
int[] Permutation(int n,Random random) {var a=Enumerable.Range(0,n).ToArray();for(int i=n-1;i>0;i--){int j=random.Next(i+1);(a[i],a[j])=(a[j],a[i]);}return a;}
var losses=new List<double>();
for(int epoch=0;epoch<60;epoch++) {double total=0;foreach(int i in Permutation(pairs.Count,rng)){var pair=pairs[i];var (loss,g)=model.Cbow(pair.Ctx,pair.Tgt);model.Update(g,0.05);total+=loss;}losses.Add(total/pairs.Count);}
Console.WriteLine($"CBOW perte moyenne : {losses[0]:F6} → {losses[^1]:F6} ; 60 époques, lr=0,05");
var svg=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='380' viewBox='0 0 760 380'><rect width='760' height='380' fill='white'/><text x='380' y='24' text-anchor='middle' font-size='16'>Convergence CBOW</text>");
double ymin=0,ymax=Math.Ceiling(losses.Max()*2)/2,left=70,top=50,width=640,height=250;
for(int j=0;j<=5;j++){double y=top+height-j*height/5;svg.Append($"<line x1='{F(left)}' y1='{F(y)}' x2='{F(left+width)}' y2='{F(y)}' stroke='#ddd'/><text x='62' y='{F(y+4)}' text-anchor='end' font-size='12'>{F(ymax*j/5)}</text>");}
for(int j=0;j<=6;j++){double x=left+j*width/6;svg.Append($"<text x='{F(x)}' y='320' text-anchor='middle' font-size='12'>{j*10}</text>");}
svg.Append("<path d='M70 50 V300 H710' fill='none' stroke='black'/><text x='70' y='42' font-size='12'>perte moyenne (cross-entropie)</text><text x='390' y='345' text-anchor='middle' font-size='12'>époque</text>");
var points=string.Join(" ",losses.Select((v,i)=>$"{F(left+(i+1)*width/60)},{F(top+height-v/ymax*height)}"));svg.Append($"<polyline points='{points}' fill='none' stroke='#2465a8' stroke-width='2'/><line x1='540' y1='352' x2='560' y2='352' stroke='#2465a8' stroke-width='2'/><text x='566' y='356' font-size='12'>CBOW</text></svg>");display(new PlotSvg(svg.ToString()));

CBOW perte moyenne : 3.538667 → 0.601730 ; 60 époques, lr=0,05


Convergence CBOW 0 0.8 1.6 2.4 3.2 4 0 10 20 30 40 50 60 perte moyenne (cross-entropie) époque <polyline points='80.667,78.833 91.333,86.059 102,90.467 112.667,92.719 123.333,94.272 134,95.316 144.667,96.873 155.333,98.343 166,100.241 176.667,102.421 187.333,105.074 198,108.444 208.667,111.526 219.333,115.668 230,119.98 240.667,124.254 251.333,129.025 262,133.601 272.667,138.665 283.333,143.605 294,148.049 304.667,153.13 315.333,157.625 326,162.343 336.667,166.35 347.333,170.984 358,174.766 368.667,178.955 379.333,183.199 390,187.07 400.667,190.708 411.333,195.147 422,198.508 432.667,202.162 443.333,205.647 454,209.142 464.667,212.283 475.333,216.439 486,219.209 496.667,222.497 507.333,224.895 518,227.826 528.667,230.452 539.333,233.284 550,236.003 560.667,238.882 571.333,240.836 582,242.634 592.667,244.933 603.333,246.807 614,248.635 624.667,250.462 635.333,252.267 646,253.898 656.667,255.54 667.333,257.075 678,258.321 688.667,259.978 699.333,260.942 710,262.392' fill='none' stroke='#2465a8' stroke-width='2'/> CBOW

### Lecture du résultat

La perte moyenne CBOW passe de **3,539 à 0,602** sur 60 époques à `lr = 0,05`. La courbe montre une baisse sur cette exécution, sans garantir cette forme pour tous les pas. La perte résiduelle dépend à la fois du corpus, du modèle et du budget ; elle n'isole pas à elle seule une cause. Chaque époque parcourt les 98 paires dans un ordre mélangé.

## 5. Les vecteurs ont-ils une structure ?

Le test de la qualité d'un espace de vecteurs : les **mots proches sémantiquement** doivent être **proches géométriquement**. On mesure la similarité par le **cosinus** des lignes de `W` (les vecteurs de mots appris).

In [8]:
double Cosine(double[] a,double[] b) {double dot=0,na=0,nb=0;for(int i=0;i<a.Length;i++){dot+=a[i]*b[i];na+=a[i]*a[i];nb+=b[i]*b[i];}return dot/(Math.Sqrt(na)*Math.Sqrt(nb)+1e-9);}
double[] VectorOf(string word) => Enumerable.Range(0,D).Select(j=>model.W[word2id[word],j]).ToArray();
(string Word,double Score)[] Neighbors(string word,int k=5) {var v=VectorOf(word);return vocab.Where(w=>w!=word).Select(w=>(Word:w,Score:Cosine(v,VectorOf(w)))).OrderByDescending(x=>x.Score).Take(k).ToArray();}
foreach(string word in new[]{"roi","chat","pomme","cheval","reine"}) Console.WriteLine($"{word,-8} → "+string.Join(", ",Neighbors(word).Select(x=>$"{x.Word} ({x.Score:F2})")));

roi      → champ (0.63), royaume (0.60), reine (0.56), jardin (0.51), trone (0.51)


chat     → chaton (0.90), du (0.70), trone (0.55), sur (0.51), chiot (0.49)


pomme    → poire (0.73), enfant (0.66), mange (0.61), fruit (0.52), arbre (0.51)


cheval   → trone (0.49), chiot (0.47), champ (0.46), jardin (0.43), jument (0.43)


reine    → poire (0.80), couronne (0.73), jument (0.65), roi (0.56), pomme (0.50)


### Lecture du résultat

Les cinq listes révèlent des voisinages variés : `chat → chaton (0,90)` et `pomme → poire (0,73)`, tandis que `roi` a notamment `champ (0,63)` et `reine (0,56)` parmi ses voisins. Les contextes communs, dont les déterminants, peuvent contribuer à ces regroupements ; ils n'isolent pas une composante sémantique unique. Sur 98 tokens, les rangs restent sensibles à l'initialisation et au protocole.

In [9]:
(string Word,double Score)[] Analogy(string a,string b,string c) {var va=VectorOf(a);var vb=VectorOf(b);var vc=VectorOf(c);var v=Enumerable.Range(0,D).Select(i=>va[i]-vb[i]+vc[i]).ToArray();return vocab.Where(w=>w!=a && w!=b && w!=c).Select(w=>(Word:w,Score:Cosine(v,VectorOf(w)))).OrderByDescending(x=>x.Score).Take(3).ToArray();}
foreach(var (a,b,c) in new[]{("roi","cheval","jument"),("chat","chaton","chiot")}) Console.WriteLine($"{a} - {b} + {c} → "+string.Join(", ",Analogy(a,b,c).Select(x=>$"{x.Word} ({x.Score:F2})")));

roi - cheval + jument → reine (0.99), poire (0.79), couronne (0.75)


chat - chaton + chiot → chien (0.75), du (0.59), trone (0.56)


### Lecture du résultat

La première analogie classe ici `reine` en tête avec un cosinus de **0,99** ; la seconde classe `chien` en tête avec **0,75**. Le corpus a été construit avec des contextes parallèles, ce qui peut contribuer à ces régularités, mais les sorties ne prouvent pas que chaque direction encode une propriété isolée. Deux exemples choisis ne mesurent pas la qualité générale des analogies.

## 6. Échantillonnage négatif

Un modèle séparé est réinitialisé sur le générateur déjà avancé. Pour chaque paire, le premier contexte est positif et cinq indices admissibles sont tirés uniformément avec remplacement. Le score utilise seulement six produits scalaires. Ce code pédagogique alloue et met toutefois à jour des tableaux denses de taille `V × D` : son coût reste en `O(VD)` par pas. Des mises à jour creuses permettraient un coût de l'ordre de `O(kD)` pour ces lignes. L'objectif SGNS diffère de la cross-entropie CBOW et leurs pertes ne sont pas comparables.

In [10]:
int[] SampleNegatives(int v,int pos,int tgt,int k,Random random) {if(k<0) throw new ArgumentOutOfRangeException(nameof(k)); var pool=Enumerable.Range(0,v).Where(i=>i!=pos && i!=tgt).ToArray();if(k>0 && pool.Length==0) throw new ArgumentException("Aucun négatif admissible");return Enumerable.Range(0,k).Select(_=>pool[random.Next(pool.Length)]).ToArray();}
if(!SampleNegatives(3,0,1,5,new Random(80)).SequenceEqual(new[]{2,2,2,2,2}) || SampleNegatives(2,0,1,0,new Random(1)).Length!=0) throw new Exception("Sampler incorrect");
try {SampleNegatives(2,0,1,1,new Random(1));throw new Exception("Domaine vide accepté");} catch(ArgumentException) {Console.WriteLine("Sampler : cardinalité et domaine validés");}
var fixtureS=new WordModel(3,1,new Normal(0,0.1,new Random(9)));fixtureS.W[0,0]=1;fixtureS.B[0]=0;fixtureS.Wp[1,0]=-30;fixtureS.Wp[2,0]=30;fixtureS.Bp[1]=fixtureS.Bp[2]=0;
var (_,gs)=fixtureS.Sgns(0,1,new[]{2,2});foreach(int row in new[]{1,2}) {double old=fixtureS.Wp[row,0];fixtureS.Wp[row,0]=old+eps;double plus=fixtureS.Sgns(0,1,new[]{2,2}).Item1;fixtureS.Wp[row,0]=old-eps;double minus=fixtureS.Sgns(0,1,new[]{2,2}).Item1;fixtureS.Wp[row,0]=old;if(Math.Abs(gs.Wp[row,0]-(plus-minus)/(2*eps))>1e-7) throw new Exception("Gradient SGNS répété incorrect");}
Console.WriteLine("SGNS extrême et négatifs répétés : différences finies validées");
model=new WordModel(V,D,normal);
double errSgns=CheckGradient(()=>model.Sgns(0,1,new[]{2,2,3,4,5}));
if(errSgns>1e-4) throw new Exception("Gradient SGNS initial incorrect");
Console.WriteLine($"SGNS initial avec doublon : erreur maximale {errSgns:E3}");
var nsLosses=new List<double>();
for(int epoch=0;epoch<60;epoch++) {double total=0;foreach(int i in Permutation(pairs.Count,rng)){var pair=pairs[i];int pos=pair.Ctx[0];var neg=SampleNegatives(V,pos,pair.Tgt,5,rng);var (loss,g)=model.Sgns(pair.Tgt,pos,neg);model.Update(g,0.05);total+=loss;}nsLosses.Add(total/pairs.Count);}
Console.WriteLine($"SGNS perte : {nsLosses[0]:F6} → {nsLosses[^1]:F6} ; 60 époques, lr=0,05, k=5");

Sampler : cardinalité et domaine validés


SGNS extrême et négatifs répétés : différences finies validées


SGNS initial avec doublon : erreur maximale 7.819E-011


SGNS perte : 3.804113 → 0.841403 ; 60 époques, lr=0,05, k=5


### Lecture du résultat

La perte SGNS passe ici de **3,804 à 0,841** en 60 époques. Elle somme un terme positif et cinq termes négatifs : son échelle et son objectif diffèrent de la cross-entropie CBOW, donc les valeurs ne se comparent pas directement. Le nouveau modèle est également réinitialisé ; les différences observées ne s'attribuent pas au seul objectif.

Le calcul des scores touche `1 + k` lignes, mais l'implémentation alloue et met à jour des matrices denses `V × D` ; elle reste en `O(VD)` par pas. Une mise à jour creuse changerait cette partie du coût. Les négatifs sont uniformes sur les indices admissibles, indépendamment de leur fréquence ; ils peuvent se répéter et chaque contribution est accumulée avant mise à jour.

In [11]:
foreach(string word in new[]{"roi","chat","pomme"}) Console.WriteLine($"{word,-8} → "+string.Join(", ",Neighbors(word).Select(x=>$"{x.Word} ({x.Score:F2})")));

roi      → reine (0.69), aboie (0.51), miaule (0.48), le (0.33), cheval (0.31)


chat     → jument (0.85), arbre (0.34), cheval (0.31), enfant (0.26), couronne (0.25)


pomme    → chiot (0.75), poire (0.54), chien (0.35), fruit (0.27), du (0.25)


### Lecture du résultat

Après SGNS, `roi → reine (0,69)`, `chat → jument (0,85)` et `pomme → chiot (0,75)` sont les premiers voisins affichés. En CBOW, `chaton` et `poire` menaient les listes de `chat` et `pomme`. Les deux objectifs et initialisations diffèrent : ces sorties montrent des géométries différentes, sans identifier la cause unique de chaque rang. Le tirage uniforme ne privilégie pas les mots fréquents.

### Exercice 2 — Échantillonnage pondéré

Tirez `k` indices selon `freq**0.75`, avec le générateur passé en argument. Une fréquence nulle ne doit jamais être tirée ; les deux fixtures concentrent la masse sur des indices différents.

Pour rendre la fixture reproductible, effectuez chaque tirage par l’inverse de la fonction de répartition : un uniforme de `generator` par indice et le premier cumul strictement supérieur à ce tirage. La masse est proportionnelle à `freq**power`.


In [12]:
// Exercice 2 : tirages proportionnels à freq^0,75 avec RNG explicite.
int[] FreqNegatives(double[] freq,int k,Random generator,double power=0.75) {
    // TODO : normaliser les poids puis tirer k fois.
    return null;
}
var ex2=FreqNegatives(new[]{0.0,1.0,0.0},6,new Random(7));
if(ex2==null) Console.WriteLine("Exercice 2 à compléter");
else {
    var autre=FreqNegatives(new[]{0.0,0.0,2.0},6,new Random(7));
    bool valide=ex2.Length==6 && autre!=null && autre.Length==6 && ex2.All(i=>i==1) && autre.All(i=>i==2);
    foreach(var (freq,seuil) in new[]{(new[]{1.0,16.0,0.0},1.0/9),(new[]{16.0,1.0,0.0},8.0/9)}) {
        var oracle=new Random(42); var attendu=Enumerable.Range(0,50).Select(_=>oracle.NextDouble()<seuil?0:1).ToArray();
        var observe=FreqNegatives(freq,50,new Random(42));
        valide=valide && observe!=null && observe.SequenceEqual(attendu);
    }
    Console.WriteLine(valide ? "Fixtures sampler validées" : "Réponse incorrecte");
}

Exercice 2 à compléter


## 7. Confrontation avec le carnet 07 (statistique)

Le carnet 07 construit des vecteurs par cooccurrence, PPMI et SVD ; ici ils sont appris par prédiction. Les corpus diffèrent : les sondes suivantes décrivent uniquement les vecteurs neuraux de ce carnet et ne constituent pas un benchmark apparié. Les signes des vecteurs singuliers et les bases dans les sous-espaces dégénérés ne sont pas uniques.

In [13]:
foreach(string word in new[]{"pomme","poire","roi","chat"}) Console.WriteLine($"[neural 08] {word,-8} → "+string.Join(", ",Neighbors(word,3).Select(x=>x.Word)));

[neural 08] pomme    → chiot, poire, chien


[neural 08] poire    → pomme, du, chien


[neural 08] roi      → reine, aboie, miaule


[neural 08] chat     → jument, arbre, cheval


### Lecture du résultat

Les quatre sondes SGNS donnent `pomme → chiot, poire, chien`, `poire → pomme, du, chien`, `roi → reine, aboie, miaule` et `chat → jument, arbre, cheval`. Des associations attendues et inattendues coexistent ; ces résultats concernent uniquement ce corpus et cette graine. Le carnet 07 utilise un autre corpus : ses sorties ne permettent pas une comparaison appariée de ces rangs. Une comparaison solide demanderait mêmes données, protocole et plusieurs initialisations.

## Exercices — récapitulatif

Les exercices répartis portent sur la [vérification du gradient skip-gram](#exercice-1--vérifier-le-gradient-skip-gram), le [sampler pondéré](#exercice-2--échantillonnage-pondéré) et le [diagnostic de stabilité](#exercice-3--rareté-et-stabilité). Ils laissent à compléter les fonctions et testent plusieurs cas distincts.

### Exercice 3 — Rareté et stabilité

Comptez les occurrences comme cible et calculez la distance entre deux vecteurs issus de deux entraînements. Une norme seule ne mesure pas la stabilité ; une occurrence dans le corpus est revue à chacune des 60 époques. Comparez plusieurs graines avant de conclure. La distance brute dépend aussi d'une rotation globale des coordonnées : alignez orthogonalement les espaces ou complétez-la par une comparaison des voisinages ou des matrices de Gram avant de parler de stabilité géométrique.

In [14]:
// Exercice 3 : occurrences-cible et distance entre deux modèles entraînés.
Dictionary<int,(int Count,double Distance)> AnalyseRares((int[] Ctx,int Tgt)[] exemples,double[,] a,double[,] b) {
    // TODO : compter les cibles et mesurer leur distance euclidienne entre deux runs.
    return null;
}
var pEx=new[]{(new[]{0},0),(new[]{0},0),(new[]{1},1)};var va=new double[,]{{0,0},{1,0}};var vb=new double[,]{{3,4},{1,2}};
var ex3=AnalyseRares(pEx,va,vb);
if(ex3==null) Console.WriteLine("Exercice 3 à compléter");
else {var autre=AnalyseRares(new[]{(new[]{1},1)},va,va);bool valid=ex3.Count==2 && ex3.ContainsKey(0) && ex3.ContainsKey(1) && autre!=null && autre.Count==1 && autre.ContainsKey(1) && ex3[0].Count==2 && ex3[1].Count==1 && double.IsFinite(ex3[0].Distance) && double.IsFinite(ex3[1].Distance) && Math.Abs(ex3[0].Distance-5)<1e-9 && Math.Abs(ex3[1].Distance-2)<1e-9 && autre[1].Count==1 && Math.Abs(autre[1].Distance)<1e-9;Console.WriteLine(valid?"Fixtures stabilité validées":"Réponse incorrecte");}

Exercice 3 à compléter


## À retenir

- Le corpus compte **98 tokens, 36 mots et 98 paires** ; la fenêtre conserve deux mots à gauche et un à droite et traverse les retours à la ligne.
- CBOW prédit le centre depuis la moyenne des vecteurs de contexte ; la perte initiale mesurée est **3,626**, proche de `ln(36)`. Les différences finies vérifient les 1 204 paramètres et un contexte répété.
- En 60 époques, la perte CBOW passe de **3,539 à 0,602**. Skip-gram prédit les voisins depuis le centre ; son entraînement softmax n'est pas réalisé ici.
- Le modèle SGNS réinitialisé passe de **3,804 à 0,841** pour son objectif distinct, avec cinq négatifs uniformes admissibles tirés avec remplacement. Les doublons sont accumulés.
- Le scoring SGNS évalue `1+k` lignes, mais cette version alloue et met à jour des gradients denses : `O(VD)` par pas. Une implémentation creuse pourrait réduire ce travail.
- Les voisinages et analogies sont des mesures sur un petit corpus, non des garanties ; la SVD du carnet 07 utilise d'autres données et ses signes ou bases ne sont pas uniques.

## Vérifiez votre compréhension

1. Quel rôle jouent les lignes de `W` et `W'` en CBOW ?
2. Pourquoi la perte SGNS ne se compare-t-elle pas directement à la perte CBOW ?
3. Pourquoi ce code SGNS ne coûte-t-il pas seulement `O(kD)` ?
4. Comment examiner la stabilité d'un mot rare ?

<details><summary>Réponses</summary>

1. `W` fournit les vecteurs d'entrée des mots de contexte ; `W'` donne les poids des sorties.
2. Les objectifs diffèrent, et le second modèle est réinitialisé.
3. Il construit et met à jour des tableaux denses `V × D`, même si seuls `1+k` scores sont calculés.
4. Compter les occurrences, puis comparer les vecteurs entre plusieurs entraînements avec graines distinctes ; une norme seule ne mesure pas cette stabilité. La distance brute dépend aussi d'une rotation globale ; alignez les espaces ou comparez les voisinages ou les matrices de Gram.

</details>

## Pour aller plus loin

- [09 — fastText](09-FastText-Subword-Vectors.ipynb) construit des représentations avec des sous-mots.
- [07 — cooccurrence, PPMI et SVD](07-Vecteurs-de-Mots-Coocc-PPMI-SVD.ipynb) présente une autre méthode sur un autre corpus.
- Mikolov et al. (2013), *Efficient Estimation of Word Representations in Vector Space* ; Goldberg & Levy (2014), *word2vec Explained*.
